In [1]:
# %%
# Cell 1: Imports and Setup
import os
import sys
import time
import pickle
import logging
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

# Display settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

print("✓ All imports successful")

✓ All imports successful


In [2]:
# %%
# Cell 2: Setup Logging & Directories
OUTPUT_DIR = r"C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Setup logging
log_file = os.path.join(RESULTS_DIR, f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

logger.info("=" * 80)
logger.info("STARTING COMPREHENSIVE MODEL TRAINING PIPELINE (MLflow ENABLED)")
logger.info("=" * 80)

print(f"✓ Logging configured: {log_file}")
print(f"✓ Results directory: {RESULTS_DIR}")

2025-12-05 14:44:56,687 - INFO - ================================================================================
2025-12-05 14:44:56,689 - INFO - STARTING COMPREHENSIVE MODEL TRAINING PIPELINE (MLflow ENABLED)
2025-12-05 14:44:56,690 - INFO - ================================================================================
2025-12-05 14:44:56,689 - INFO - STARTING COMPREHENSIVE MODEL TRAINING PIPELINE (MLflow ENABLED)
2025-12-05 14:44:56,690 - INFO - ================================================================================
✓ Logging configured: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results\training_log_20251205_144456.log
✓ Results directory: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results
✓ Logging configured: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results\training_log_20251205_144456.log
✓ Results direc

In [3]:
# %%
# Cell 3: MLflow configuration
# Set an experiment name. If it does not exist, MLflow will create it.
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'Forest_Audio_Experiments')
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', None)  # e.g. 'http://127.0.0.1:5000' or None for local mlruns

if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
logger.info(f"MLflow experiment set: {MLFLOW_EXPERIMENT_NAME} | tracking_uri: {mlflow.get_tracking_uri()}")

2025/12/05 14:44:56 INFO mlflow.tracking.fluent: Experiment with name 'Forest_Audio_Experiments' does not exist. Creating a new experiment.


2025-12-05 14:44:56,769 - INFO - MLflow experiment set: Forest_Audio_Experiments | tracking_uri: file:///c:/Users/Lenovo/Documents/projects/research/Major-Project/Forest-Audio/code/mlruns


In [4]:
# %%
# Cell 4: Configuration
MFCC_LIST = [8, 16, 32, 64, 128]
RANDOM_SEED = 42

# Binary mapping for your dataset
binary_mapping = {
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 0, 9: 1,
    10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 0,
    18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 0,
    26: 0, 27: 0
}

logger.info(f"MFCC configurations to test: {MFCC_LIST}")
logger.info(f"Random seed: {RANDOM_SEED}")
logger.info(f"Binary mapping: {binary_mapping}")

2025-12-05 14:44:56,782 - INFO - MFCC configurations to test: [8, 16, 32, 64, 128]
2025-12-05 14:44:56,786 - INFO - Random seed: 42
2025-12-05 14:44:56,787 - INFO - Binary mapping: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 0, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 0, 18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 0, 26: 0, 27: 0}
2025-12-05 14:44:56,786 - INFO - Random seed: 42
2025-12-05 14:44:56,787 - INFO - Binary mapping: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 0, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 0, 18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 0, 26: 0, 27: 0}


In [5]:
# %%
# Cell 5: Define Model Configurations with Hyperparameters
param_grids = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED),
        "params": {
            "C": [0.01, 0.1, 1, 10],
            "solver": ["liblinear", "lbfgs"],
            "penalty": ["l2"]
        }
    },
    "Ridge Classifier": {
        "model": RidgeClassifier(class_weight="balanced", random_state=RANDOM_SEED),
        "params": {
            "alpha": [0.1, 1.0, 10.0],
            "solver": ["auto", "saga"]
        }
    },
    "SVM (RBF)": {
        "model": SVC(class_weight="balanced", probability=True, random_state=RANDOM_SEED),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["rbf"],
            "gamma": ["scale", "auto"]
        }
    },
    "Linear SVM": {
        "model": LinearSVC(class_weight="balanced", max_iter=2000, random_state=RANDOM_SEED),
        "params": {
            "C": [0.1, 1, 10],
            "loss": ["hinge", "squared_hinge"]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }
    },
    "Extra Trees": {
        "model": ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_SEED),
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.1, 0.2],
            "max_depth": [3, 5, 7],
            "subsample": [0.8, 1.0]
        }
    },
    "XGBoost": {
        "model": XGBClassifier(
            use_label_encoder=False, 
            eval_metric="logloss", 
            random_state=RANDOM_SEED,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6, 10],
            "learning_rate": [0.01, 0.1, 0.2],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "scale_pos_weight": [1, 2]
        }
    },
    "LightGBM": {
        "model": LGBMClassifier(
            class_weight="balanced", 
            random_state=RANDOM_SEED, 
            force_col_wise=True,
            verbose=-1
        ),
        "params": {
            "n_estimators": [100, 200],
            "num_leaves": [31, 63],
            "learning_rate": [0.01, 0.1, 0.2],
            "min_child_samples": [20, 50]
        }
    },
    "AdaBoost": {
        "model": AdaBoostClassifier(random_state=RANDOM_SEED),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 1.0]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_SEED),
        "params": {
            "max_depth": [5, 10, 20, None],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"]
        }
    },
    "K-Nearest Neighbors": {
        "model": KNeighborsClassifier(n_jobs=-1),
        "params": {
            "n_neighbors": [3, 5, 7, 9],
            "weights": ["uniform", "distance"],
            "metric": ["euclidean", "manhattan"]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {
            "var_smoothing": [1e-9, 1e-8, 1e-7]
        }
    },
    "MLP Neural Network": {
        "model": MLPClassifier(max_iter=1000, random_state=RANDOM_SEED),
        "params": {
            "hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "activation": ["relu", "tanh"],
            "alpha": [0.0001, 0.001],
            "learning_rate": ["constant", "adaptive"]
        }
    }
}

logger.info(f"Total models configured: {len(param_grids)}")
for model_name in param_grids.keys():
    logger.info(f"  - {model_name}")

2025-12-05 14:44:56,814 - INFO - Total models configured: 14
2025-12-05 14:44:56,818 - INFO -   - Logistic Regression
2025-12-05 14:44:56,819 - INFO -   - Ridge Classifier
2025-12-05 14:44:56,822 - INFO -   - SVM (RBF)
2025-12-05 14:44:56,818 - INFO -   - Logistic Regression
2025-12-05 14:44:56,819 - INFO -   - Ridge Classifier
2025-12-05 14:44:56,822 - INFO -   - SVM (RBF)
2025-12-05 14:44:56,823 - INFO -   - Linear SVM
2025-12-05 14:44:56,825 - INFO -   - Random Forest
2025-12-05 14:44:56,826 - INFO -   - Extra Trees
2025-12-05 14:44:56,828 - INFO -   - Gradient Boosting
2025-12-05 14:44:56,829 - INFO -   - XGBoost
2025-12-05 14:44:56,831 - INFO -   - LightGBM
2025-12-05 14:44:56,823 - INFO -   - Linear SVM
2025-12-05 14:44:56,825 - INFO -   - Random Forest
2025-12-05 14:44:56,826 - INFO -   - Extra Trees
2025-12-05 14:44:56,828 - INFO -   - Gradient Boosting
2025-12-05 14:44:56,829 - INFO -   - XGBoost
2025-12-05 14:44:56,831 - INFO -   - LightGBM
2025-12-05 14:44:56,833 - INFO -   

In [6]:
# %%
# Cell 5b: Utility Functions

def get_model_size(model):
    """Calculate model size in MB"""
    import io
    buffer = io.BytesIO()
    pickle.dump(model, buffer)
    size_bytes = buffer.tell()
    size_mb = size_bytes / (1024 * 1024)
    return size_mb


def measure_inference_time(model, X_test, n_runs=100):
    """Measure average inference time"""
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model.predict(X_test[:100])  # predict on 100 samples
        end = time.perf_counter()
        times.append(end - start)
    return np.mean(times) * 1000  # in milliseconds


def calculate_metrics(y_true, y_pred, y_proba=None):
    """Calculate comprehensive metrics"""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }
    
    if y_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
        except:
            metrics['roc_auc'] = np.nan
    else:
        metrics['roc_auc'] = np.nan
    
    return metrics

print("✓ Utility functions defined")

✓ Utility functions defined


In [7]:
# %%
# Cell 6: Main Training Loop with MLflow logging
all_results = []

logger.info("\n" + "=" * 80)
logger.info("STARTING TRAINING LOOP")
logger.info("=" * 80)

# Outer loop: iterate over MFCC configurations
for n_mfcc in tqdm(MFCC_LIST, desc="MFCC Configs", position=0):
    logger.info(f"\n{'='*80}")
    logger.info(f"PROCESSING n_mfcc = {n_mfcc}")
    logger.info(f"{'='*80}")
    
    # Load data for this MFCC configuration
    try:
        X_train = np.load(os.path.join(OUTPUT_DIR, f"X_train_{n_mfcc}.npy"))
        X_test = np.load(os.path.join(OUTPUT_DIR, f"X_test_{n_mfcc}.npy"))
        y_train = np.load(os.path.join(OUTPUT_DIR, "y_train.npy"))
        y_test = np.load(os.path.join(OUTPUT_DIR, "y_test.npy"))
        
        logger.info(f"Data loaded - X_train: {X_train.shape}, X_test: {X_test.shape}")
    except FileNotFoundError as e:
        logger.error(f"File not found for n_mfcc={n_mfcc}: {e}")
        continue
    
    # Apply binary mapping
    y_train_bin = np.array([binary_mapping[label] for label in y_train])
    y_test_bin = np.array([binary_mapping[label] for label in y_test])
    
    class_dist = np.bincount(y_train_bin)
    logger.info(f"Binary class distribution - Class 0: {class_dist[0]}, Class 1: {class_dist[1]}")

    # Log dataset-level info to MLflow as run tags (will be updated with each run)
    dataset_info = {
        'X_train_shape': str(X_train.shape),
        'X_test_shape': str(X_test.shape),
        'y_train_shape': str(y_train_bin.shape),
        'y_test_shape': str(y_test_bin.shape),
        'class_0_count': int(class_dist[0]),
        'class_1_count': int(class_dist[1]),
        'n_mfcc': int(n_mfcc)
    }
    
    # Inner loop: iterate over models
    for model_name, config in tqdm(param_grids.items(), 
                                    desc=f"Models (n_mfcc={n_mfcc})", 
                                    position=1, 
                                    leave=False):
        logger.info(f"\n--- Training {model_name} (n_mfcc={n_mfcc}) ---")
        
        try:
            # GridSearch with cross-validation
            grid = GridSearchCV(
                estimator=config["model"],
                param_grid=config["params"],
                scoring="f1",
                cv=3,
                n_jobs=-1,
                verbose=0
            )
            
            # Start an MLflow run per model+mfcc combination
            with mlflow.start_run(run_name=f"{model_name}_mfcc{n_mfcc}") as run:
                mlflow.set_tag("model_name", model_name)
                mlflow.set_tag("n_mfcc", n_mfcc)
                mlflow.set_tags({f"dataset_{k}": v for k, v in dataset_info.items()})
                mlflow.log_param("random_seed", RANDOM_SEED)

                # Training
                train_start = time.perf_counter()
                grid.fit(X_train, y_train_bin)
                train_end = time.perf_counter()
                train_time = train_end - train_start
                
                best_model = grid.best_estimator_
                best_params = grid.best_params_
                
                logger.info(f"Best params: {best_params}")
                logger.info(f"Training time: {train_time:.2f}s")

                # Log search/grid info
                mlflow.log_param("best_params", str(best_params))
                mlflow.log_metric("cv_best_score", float(grid.best_score_))
                mlflow.log_metric("train_time_sec", float(train_time))

                # Predictions
                y_pred = best_model.predict(X_test)
                
                # Get probability predictions if available
                y_proba = None
                if hasattr(best_model, "predict_proba"):
                    y_proba = best_model.predict_proba(X_test)[:, 1]
                elif hasattr(best_model, "decision_function"):
                    y_proba = best_model.decision_function(X_test)
                
                # Calculate metrics
                metrics = calculate_metrics(y_test_bin, y_pred, y_proba)

                # Model size
                model_size = get_model_size(best_model)
                
                # Inference time
                inference_time = measure_inference_time(best_model, X_test)

                logger.info(f"Accuracy: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")
                logger.info(f"Model size: {model_size:.4f} MB")
                logger.info(f"Inference time: {inference_time:.4f} ms (per 100 samples)")

                # Log metrics to MLflow
                mlflow.log_metrics({k: float(v) if not np.isnan(v) else None for k, v in metrics.items()})
                mlflow.log_metric("model_size_mb", float(model_size))
                mlflow.log_metric("inference_time_ms_per_100_samples", float(inference_time))

                # Save model artifact to disk and to MLflow
                model_filename = os.path.join(RESULTS_DIR, f"model_{model_name.replace(' ', '_')}_{n_mfcc}.pkl")
                with open(model_filename, 'wb') as f:
                    pickle.dump(best_model, f)
                mlflow.log_artifact(model_filename, artifact_path="model_pickle")

                # Also use mlflow.sklearn to log the model (allows later loading)
                try:
                    mlflow.sklearn.log_model(best_model, artifact_path="sklearn_model")
                except Exception as e:
                    logger.warning(f"mlflow.sklearn.log_model failed: {e}")

                # Confusion matrix plot and artifact
                try:
                    cm = confusion_matrix(y_test_bin, y_pred)
                    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
                    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm)
                    ax_cm.set_xlabel('Predicted')
                    ax_cm.set_ylabel('Actual')
                    ax_cm.set_title(f'Confusion Matrix: {model_name} (n_mfcc={n_mfcc})')
                    cm_path = os.path.join(RESULTS_DIR, f"cm_{model_name.replace(' ', '_')}_{n_mfcc}.png")
                    plt.savefig(cm_path, dpi=200, bbox_inches='tight')
                    plt.close(fig_cm)
                    mlflow.log_artifact(cm_path, artifact_path='plots')
                except Exception as e:
                    logger.warning(f"Could not create confusion matrix plot: {e}")

                # If available, log feature importances for tree-based models
                try:
                    if hasattr(best_model, 'feature_importances_'):
                        fi = best_model.feature_importances_
                        fi_path = os.path.join(RESULTS_DIR, f"feature_importances_{model_name.replace(' ', '_')}_{n_mfcc}.csv")
                        pd.DataFrame({'feature_index': np.arange(len(fi)), 'importance': fi}).to_csv(fi_path, index=False)
                        mlflow.log_artifact(fi_path, artifact_path='feature_importances')
                except Exception as e:
                    logger.warning(f"Could not extract feature importances: {e}")

                # Record other useful run-level info
                run_id = run.info.run_id
                mlflow.set_tag('run_id', run_id)
                mlflow.log_param('model_pickle_path', model_filename)

                # Append to results list
                result = {
                    'n_mfcc': n_mfcc,
                    'model_name': model_name,
                    'accuracy': metrics['accuracy'],
                    'precision': metrics['precision'],
                    'recall': metrics['recall'],
                    'f1': metrics['f1'],
                    'roc_auc': metrics['roc_auc'],
                    'train_time_sec': train_time,
                    'inference_time_ms': inference_time,
                    'model_size_mb': model_size,
                    'best_params': str(best_params),
                    'cv_best_score': grid.best_score_,
                    'mlflow_run_id': run_id
                }
                all_results.append(result)

        except Exception as e:
            logger.error(f"Error training {model_name} with n_mfcc={n_mfcc}: {e}")
            continue

logger.info("\n" + "=" * 80)
logger.info("TRAINING COMPLETED")
logger.info("=" * 80)

# Convert to DataFrame
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, "all_results.csv"), index=False)
logger.info(f"Results saved to: {os.path.join(RESULTS_DIR, 'all_results.csv')}")

print("\n✓ Training loop completed!")
print(f"Total experiments: {len(results_df)}")

2025-12-05 14:44:56,913 - INFO - 
2025-12-05 14:44:56,914 - INFO - STARTING TRAINING LOOP
2025-12-05 14:44:56,914 - INFO - STARTING TRAINING LOOP
2025-12-05 14:44:56,917 - INFO - ================================================================================
2025-12-05 14:44:56,917 - INFO - ================================================================================


MFCC Configs:   0%|          | 0/5 [00:00<?, ?it/s]

2025-12-05 14:44:56,938 - INFO - 
2025-12-05 14:44:56,941 - INFO - PROCESSING n_mfcc = 8
2025-12-05 14:44:56,942 - INFO - ================================================================================
2025-12-05 14:44:56,949 - INFO - Data loaded - X_train: (1620, 8), X_test: (405, 8)
2025-12-05 14:44:56,952 - INFO - Binary class distribution - Class 0: 1080, Class 1: 540
2025-12-05 14:44:56,941 - INFO - PROCESSING n_mfcc = 8
2025-12-05 14:44:56,942 - INFO - ================================================================================
2025-12-05 14:44:56,949 - INFO - Data loaded - X_train: (1620, 8), X_test: (405, 8)
2025-12-05 14:44:56,952 - INFO - Binary class distribution - Class 0: 1080, Class 1: 540


Models (n_mfcc=8):   0%|          | 0/14 [00:00<?, ?it/s]

2025-12-05 14:44:56,971 - INFO - 
--- Training Logistic Regression (n_mfcc=8) ---
2025-12-05 14:45:12,315 - INFO - Best params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
2025-12-05 14:45:12,321 - INFO - Training time: 14.93s
2025-12-05 14:45:12,315 - INFO - Best params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
2025-12-05 14:45:12,321 - INFO - Training time: 14.93s
2025-12-05 14:45:12,687 - INFO - Accuracy: 0.6667, F1: 0.5687
2025-12-05 14:45:12,689 - INFO - Model size: 0.0007 MB
2025-12-05 14:45:12,693 - INFO - Inference time: 0.3489 ms (per 100 samples)
2025-12-05 14:45:12,687 - INFO - Accuracy: 0.6667, F1: 0.5687
2025-12-05 14:45:12,689 - INFO - Model size: 0.0007 MB
2025-12-05 14:45:12,693 - INFO - Inference time: 0.3489 ms (per 100 samples)


...

2025-12-05 15:25:21,480 - INFO - 
--- Training Naive Bayes (n_mfcc=128) ---
2025-12-05 15:25:21,619 - INFO - Best params: {'var_smoothing': 1e-09}
2025-12-05 15:25:21,620 - INFO - Training time: 0.04s
2025-12-05 15:25:21,619 - INFO - Best params: {'var_smoothing': 1e-09}
2025-12-05 15:25:21,620 - INFO - Training time: 0.04s
2025-12-05 15:25:21,733 - INFO - Accuracy: 0.7160, F1: 0.4335
2025-12-05 15:25:21,734 - INFO - Model size: 0.0045 MB
2025-12-05 15:25:21,734 - INFO - Inference time: 0.3051 ms (per 100 samples)
2025-12-05 15:25:21,733 - INFO - Accuracy: 0.7160, F1: 0.4335
2025-12-05 15:25:21,734 - INFO - Model size: 0.0045 MB
2025-12-05 15:25:21,734 - INFO - Inference time: 0.3051 ms (per 100 samples)


2025/12/05 15:25:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/05 15:25:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/12/05 15:25:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


2025-12-05 15:25:27,522 - INFO - 
--- Training MLP Neural Network (n_mfcc=128) ---
2025-12-05 15:26:08,279 - INFO - Best params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (128,), 'learning_rate': 'constant'}
2025-12-05 15:26:08,280 - INFO - Training time: 40.64s
2025-12-05 15:26:08,279 - INFO - Best params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (128,), 'learning_rate': 'constant'}
2025-12-05 15:26:08,280 - INFO - Training time: 40.64s
2025-12-05 15:26:08,397 - INFO - Accuracy: 0.7926, F1: 0.6934
2025-12-05 15:26:08,398 - INFO - Model size: 0.2659 MB
2025-12-05 15:26:08,398 - INFO - Inference time: 0.4163 ms (per 100 samples)
2025-12-05 15:26:08,397 - INFO - Accuracy: 0.7926, F1: 0.6934
2025-12-05 15:26:08,398 - INFO - Model size: 0.2659 MB
2025-12-05 15:26:08,398 - INFO - Inference time: 0.4163 ms (per 100 samples)


2025/12/05 15:26:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/05 15:26:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/12/05 15:26:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


2025-12-05 15:26:14,263 - INFO - 
2025-12-05 15:26:14,265 - INFO - TRAINING COMPLETED
2025-12-05 15:26:14,267 - INFO - ================================================================================
2025-12-05 15:26:14,281 - INFO - Results saved to: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results\all_results.csv
2025-12-05 15:26:14,265 - INFO - TRAINING COMPLETED
2025-12-05 15:26:14,267 - INFO - ================================================================================
2025-12-05 15:26:14,281 - INFO - Results saved to: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results\all_results.csv

✓ Training loop completed!
Total experiments: 70

✓ Training loop completed!
Total experiments: 70


In [8]:
# %%
# Cell 7: Results Summary (same as original but logs to MLflow as artifact)
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

display(results_df.head(20))

# Basic statistics
print("\nBasic Statistics:")
print(results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].describe())

# Save CSV and log to MLflow
summary_csv = os.path.join(RESULTS_DIR, "all_results_summary.csv")
results_df.to_csv(summary_csv, index=False)
mlflow.log_artifact(summary_csv, artifact_path='summaries')


RESULTS SUMMARY


,n_mfcc,model_name,accuracy,precision,recall,f1,roc_auc,train_time_sec,inference_time_ms,model_size_mb,best_params,cv_best_score,mlflow_run_id
0,8,Logistic Regression,0.666667,0.500000,0.659259,0.568690,0.728038,14.925484,0.348944,0.000748,"{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.583848,c7d30579b6b24db2945091a167fb57ed
1,8,Ridge Classifier,0.654321,0.486631,0.674074,0.565217,0.727380,0.064142,0.083793,0.000733,"{'alpha': 10.0, 'solver': 'auto'}",0.584570,97a33bc0f8d7497ca1879fab5b262199
2,8,SVM (RBF),0.814815,0.692308,0.800000,0.742268,0.873114,1.984435,6.287858,0.066103,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.703533,b20de24ee31b46128952f10ecc758e79
3,8,Linear SVM,0.659259,0.491803,0.666667,0.566038,0.725021,0.066210,0.105074,0.000630,"{'C': 1, 'loss': 'hinge'}",0.583434,7b343a2f44bf4a229295f77b06d7fa0e
4,8,Random Forest,0.814815,0.750000,0.666667,0.705882,0.859890,16.659018,37.966285,1.533546,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_...",0.658185,34625c26837542af8aec899f0b81022a
5,8,Extra Trees,0.809877,0.745763,0.651852,0.695652,0.888587,3.143790,55.483500,12.708545,"{'max_depth': 20, 'min_samples_split': 5, 'n_e...",0.697898,7175b6293106450daa4e5834f3001367
6,8,Gradient Boosting,0.800000,0.754717,0.592593,0.663900,0.849630,25.622895,1.295307,1.939754,"{'learning_rate': 0.2, 'max_depth': 7, 'n_esti...",0.668980,b941d63f665042ef82e0e3414cfcde45
7,8,XGBoost,0.802469,0.716535,0.674074,0.694656,0.869273,17.322090,1.076185,0.261510,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.703091,59003e7b269147ad812cae7747523429
8,8,LightGBM,0.807407,0.747826,0.637037,0.688000,0.863292,12.896653,1.561612,0.543365,"{'learning_rate': 0.1, 'min_child_samples': 50...",0.689974,c0cbb1f60eb24268881d2fa896f9dd1c
9,8,AdaBoost,0.740741,0.656250,0.466667,0.545455,0.783745,3.042124,18.775576,0.106836,"{'learning_rate': 1.0, 'n_estimators': 200}",0.557442,ebec8500e2e44e6299fd33ba02ba5bfe



Basic Statistics:
        accuracy  precision     recall         f1    roc_auc
count  70.000000  70.000000  70.000000  70.000000  70.000000
mean    0.764586   0.665025   0.660317   0.652266   0.821691
std     0.063811   0.111446   0.101583   0.081333   0.060908
min     0.639506   0.470588   0.296296   0.412371   0.696598
25%     0.704321   0.542912   0.631481   0.592573   0.764396
50%     0.793827   0.692688   0.685185   0.678710   0.849348
75%     0.814815   0.745163   0.711111   0.710573   0.872709
max     0.861728   0.876289   0.807407   0.779528   0.908395


In [9]:
# %%
# Cell 8: Top Performing Models Overall
print("\n" + "="*80)
print("TOP 10 MODELS BY F1 SCORE")
print("="*80)

top_models = results_df.nlargest(10, 'f1')[
    ['model_name', 'n_mfcc', 'f1', 'accuracy', 'precision', 'recall', 
     'roc_auc', 'model_size_mb', 'train_time_sec', 'inference_time_ms']
]
display(top_models)

top_models.to_csv(os.path.join(RESULTS_DIR, "top_10_models.csv"), index=False)
mlflow.log_artifact(os.path.join(RESULTS_DIR, "top_10_models.csv"), artifact_path='summaries')


TOP 10 MODELS BY F1 SCORE


,model_name,n_mfcc,f1,accuracy,precision,recall,roc_auc,model_size_mb,train_time_sec,inference_time_ms
50,LightGBM,64,0.779528,0.861728,0.831933,0.733333,0.908395,0.671804,26.638164,2.969795
64,LightGBM,128,0.764940,0.854321,0.827586,0.711111,0.902524,0.571819,41.180503,2.391039
30,SVM (RBF),32,0.749141,0.819753,0.698718,0.807407,0.879698,0.228390,2.353132,8.284985
33,Extra Trees,32,0.748971,0.849383,0.842593,0.674074,0.895199,5.190002,6.173962,37.234076
36,LightGBM,32,0.746154,0.837037,0.776000,0.718519,0.904883,0.667721,8.513136,1.818632
27,MLP Neural Network,16,0.746032,0.841975,0.803419,0.696296,0.881481,0.065411,100.876922,0.149062
63,XGBoost,128,0.745387,0.829630,0.742647,0.748148,0.880439,0.224367,344.594041,1.312312
2,SVM (RBF),8,0.742268,0.814815,0.692308,0.800000,0.873114,0.066103,1.984435,6.287858
41,MLP Neural Network,32,0.737226,0.822222,0.726619,0.748148,0.884883,0.088585,75.571061,0.195656
35,XGBoost,32,0.736059,0.824691,0.738806,0.733333,0.889602,0.266471,74.725933,0.906569


In [10]:
# %%
# Cell 9: Best Model Per MFCC Configuration
print("\n" + "="*80)
print("BEST MODEL FOR EACH MFCC CONFIGURATION")
print("="*80)

best_per_mfcc = results_df.loc[results_df.groupby('n_mfcc')['f1'].idxmax()][
    ['n_mfcc', 'model_name', 'f1', 'accuracy', 'precision', 'recall', 
     'roc_auc', 'model_size_mb', 'train_time_sec', 'best_params']
]
display(best_per_mfcc)

best_per_mfcc.to_csv(os.path.join(RESULTS_DIR, "best_per_mfcc.csv"), index=False)
mlflow.log_artifact(os.path.join(RESULTS_DIR, "best_per_mfcc.csv"), artifact_path='summaries')


BEST MODEL FOR EACH MFCC CONFIGURATION


,n_mfcc,model_name,f1,accuracy,precision,recall,roc_auc,model_size_mb,train_time_sec,best_params
2,8,SVM (RBF),0.742268,0.814815,0.692308,0.800000,0.873114,0.066103,1.984435,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}"
27,16,MLP Neural Network,0.746032,0.841975,0.803419,0.696296,0.881481,0.065411,100.876922,"{'activation': 'relu', 'alpha': 0.001, 'hidden..."
30,32,SVM (RBF),0.749141,0.819753,0.698718,0.807407,0.879698,0.228390,2.353132,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}"
50,64,LightGBM,0.779528,0.861728,0.831933,0.733333,0.908395,0.671804,26.638164,"{'learning_rate': 0.1, 'min_child_samples': 20..."
64,128,LightGBM,0.764940,0.854321,0.827586,0.711111,0.902524,0.571819,41.180503,"{'learning_rate': 0.2, 'min_child_samples': 50..."


In [11]:
# %%
# Cell 10: Best Model Per Algorithm
print("\n" + "="*80)
print("BEST MFCC CONFIGURATION FOR EACH MODEL")
print("="*80)

best_per_model = results_df.loc[results_df.groupby('model_name')['f1'].idxmax()][
    ['model_name', 'n_mfcc', 'f1', 'accuracy', 'precision', 'recall', 
     'roc_auc', 'model_size_mb', 'train_time_sec', 'best_params']
]
display(best_per_model)

best_per_model.to_csv(os.path.join(RESULTS_DIR, "best_per_model.csv"), index=False)
mlflow.log_artifact(os.path.join(RESULTS_DIR, "best_per_model.csv"), artifact_path='summaries')


BEST MFCC CONFIGURATION FOR EACH MODEL


,model_name,n_mfcc,f1,accuracy,precision,recall,roc_auc,model_size_mb,train_time_sec,best_params
65,AdaBoost,128,0.645418,0.780247,0.698276,0.600000,0.831248,0.106836,25.937217,"{'learning_rate': 1.0, 'n_estimators': 200}"
38,Decision Tree,32,0.648649,0.743210,0.596273,0.711111,0.762579,0.015489,1.065058,"{'criterion': 'gini', 'max_depth': 10, 'min_sa..."
33,Extra Trees,32,0.748971,0.849383,0.842593,0.674074,0.895199,5.190002,6.173962,"{'max_depth': 20, 'min_samples_split': 5, 'n_e..."
48,Gradient Boosting,64,0.732759,0.846914,0.876289,0.629630,0.886941,0.836589,106.749905,"{'learning_rate': 0.1, 'max_depth': 5, 'n_esti..."
11,K-Nearest Neighbors,8,0.730038,0.824691,0.750000,0.711111,0.842826,0.183795,0.392024,"{'metric': 'euclidean', 'n_neighbors': 3, 'wei..."
50,LightGBM,64,0.779528,0.861728,0.831933,0.733333,0.908395,0.671804,26.638164,"{'learning_rate': 0.1, 'min_child_samples': 20..."
59,Linear SVM,128,0.628205,0.713580,0.553672,0.725926,0.783841,0.001549,1.974754,"{'C': 10, 'loss': 'hinge'}"
56,Logistic Regression,128,0.613419,0.701235,0.539326,0.711111,0.783100,0.001677,0.446239,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline..."
27,MLP Neural Network,16,0.746032,0.841975,0.803419,0.696296,0.881481,0.065411,100.876922,"{'activation': 'relu', 'alpha': 0.001, 'hidden..."
54,Naive Bayes,64,0.566434,0.693827,0.536424,0.600000,0.763594,0.002539,0.026146,{'var_smoothing': 1e-09}


In [12]:
# %%
# (Remaining visualization cells identical to original) - they create plots and log them
# For brevity these cells will generate the same set of plots and log to MLflow as artifacts.

# Example: generate performance heatmaps and log
try:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]
        pivot = results_df.pivot_table(
            values=metric, 
            index='model_name', 
            columns='n_mfcc', 
            aggfunc='mean'
        )
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, cbar_kws={'label': metric})
        ax.set_title(f'{metric.upper()} Across Models and MFCC Configs', fontsize=12, fontweight='bold')
        ax.set_xlabel('n_mfcc')
        ax.set_ylabel('Model')

    ax = axes[1, 2]
    pivot_size = results_df.pivot_table(
        values='model_size_mb', 
        index='model_name', 
        columns='n_mfcc', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_size, annot=True, fmt='.2f', cmap='Reds', ax=ax, cbar_kws={'label': 'MB'})
    ax.set_title('Model Size (MB) Across Configs', fontsize=12, fontweight='bold')
    ax.set_xlabel('n_mfcc')
    ax.set_ylabel('Model')

    plt.tight_layout()
    heatmap_path = os.path.join(RESULTS_DIR, "performance_heatmaps.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    mlflow.log_artifact(heatmap_path, artifact_path='plots')
except Exception as e:
    logger.warning(f"Could not create heatmaps: {e}")

# You can re-run remaining visualization cells from the original script; ensure their generated files are saved into RESULTS_DIR

In [13]:
# %%
# Cell Final: Final Recommendations & Summary Report
print("\n" + "="*100)
print("FINAL RECOMMENDATIONS")
print("="*100)

recommendations = {
    "Best Overall Performance": results_df.nlargest(1, 'f1'),
    "Best Accuracy": results_df.nlargest(1, 'accuracy'),
    "Fastest Training": results_df.nsmallest(1, 'train_time_sec'),
    "Smallest Model": results_df.nsmallest(1, 'model_size_mb'),
    "Fastest Inference": results_df.nsmallest(1, 'inference_time_ms'),
    "Best Time Efficiency": results_df.nlargest(1, 'efficiency') if 'efficiency' in results_df.columns else results_df.nlargest(1, 'f1'),
    "Best Size Efficiency": results_df.nlargest(1, 'size_efficiency') if 'size_efficiency' in results_df.columns else results_df.nlargest(1, 'f1')
}

for category, model_row in recommendations.items():
    print(f"\n{category}:")
    print("-" * 100)
    model_info = model_row.iloc[0]
    print(f"  Model: {model_info['model_name']}")
    print(f"  n_mfcc: {model_info['n_mfcc']}")
    print(f"  F1 Score: {model_info['f1']:.4f}")
    print(f"  Accuracy: {model_info['accuracy']:.4f}")
    print(f"  Training Time: {model_info['train_time_sec']:.2f} sec")
    print(f"  Model Size: {model_info['model_size_mb']:.4f} MB")
    print(f"  Inference Time: {model_info['inference_time_ms']:.4f} ms")
    print(f"  Best Params: {model_info['best_params']}")

# Save recommendations
recommendations_df = pd.concat([df.assign(Category=cat) for cat, df in recommendations.items()])
recommendations_df.to_csv(os.path.join(RESULTS_DIR, "final_recommendations.csv"), index=False)
mlflow.log_artifact(os.path.join(RESULTS_DIR, "final_recommendations.csv"), artifact_path='summaries')

# Summary report
summary_report = {
    "Total Experiments": len(results_df),
    "Models Tested": results_df['model_name'].nunique(),
    "MFCC Configurations": len(MFCC_LIST),
    "Best Overall F1": results_df['f1'].max(),
    "Best Overall Accuracy": results_df['accuracy'].max(),
    "Average F1": results_df['f1'].mean(),
    "Average Accuracy": results_df['accuracy'].mean(),
    "Total Training Time (hours)": results_df['train_time_sec'].sum() / 3600,
    "Average Training Time (sec)": results_df['train_time_sec'].mean(),
    "Average Model Size (MB)": results_df['model_size_mb'].mean(),
    "Average Inference Time (ms)": results_df['inference_time_ms'].mean()
}

summary_df = pd.DataFrame([summary_report]).T
summary_df.columns = ['Value']

display(summary_df)

summary_path = os.path.join(RESULTS_DIR, "summary_report.csv")
summary_df.to_csv(summary_path)
mlflow.log_artifact(summary_path, artifact_path='summaries')

print(f"\n{'='*100}")
print("ALL RESULTS SAVED TO:")
print(f"  Directory: {RESULTS_DIR}")
print(f"  Main Results: all_results.csv")
print(f"  Top Models: top_10_models.csv")
print(f"  Best per MFCC: best_per_mfcc.csv")
print(f"  Best per Model: best_per_model.csv")
print(f"  Recommendations: final_recommendations.csv")
print(f"  Pareto Optimal: pareto_optimal_models.csv (if generated)")
print(f"  Summary Report: summary_report.csv")
print(f"  Log File: {log_file}")
print(f"{'='*100}")

logger.info("\n" + "=" * 80)
logger.info("PIPELINE COMPLETED SUCCESSFULLY (MLflow run metadata available)")
logger.info(f"Total experiments: {len(results_df)}")
logger.info(f"Results saved to: {RESULTS_DIR}")
logger.info("=" * 80)


FINAL RECOMMENDATIONS

Best Overall Performance:
----------------------------------------------------------------------------------------------------
  Model: LightGBM
  n_mfcc: 64
  F1 Score: 0.7795
  Accuracy: 0.8617
  Training Time: 26.64 sec
  Model Size: 0.6718 MB
  Inference Time: 2.9698 ms
  Best Params: {'learning_rate': 0.1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}

Best Accuracy:
----------------------------------------------------------------------------------------------------
  Model: LightGBM
  n_mfcc: 64
  F1 Score: 0.7795
  Accuracy: 0.8617
  Training Time: 26.64 sec
  Model Size: 0.6718 MB
  Inference Time: 2.9698 ms
  Best Params: {'learning_rate': 0.1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}

Fastest Training:
----------------------------------------------------------------------------------------------------
  Model: Naive Bayes
  n_mfcc: 8
  F1 Score: 0.4249
  Accuracy: 0.7259
  Training Time: 0.03 sec
  Model Size: 0

,Value
Total Experiments,70.000000
Models Tested,14.000000
MFCC Configurations,5.000000
Best Overall F1,0.779528
Best Overall Accuracy,0.861728
Average F1,0.652266
Average Accuracy,0.764586
Total Training Time (hours),0.539141
Average Training Time (sec),27.727277
Average Model Size (MB),0.929761



ALL RESULTS SAVED TO:
  Directory: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results
  Main Results: all_results.csv
  Top Models: top_10_models.csv
  Best per MFCC: best_per_mfcc.csv
  Best per Model: best_per_model.csv
  Recommendations: final_recommendations.csv
  Pareto Optimal: pareto_optimal_models.csv (if generated)
  Summary Report: summary_report.csv
  Log File: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\model_results\training_log_20251205_144456.log
2025-12-05 15:26:20,764 - INFO - 
2025-12-05 15:26:20,766 - INFO - PIPELINE COMPLETED SUCCESSFULLY (MLflow run metadata available)
2025-12-05 15:26:20,766 - INFO - PIPELINE COMPLETED SUCCESSFULLY (MLflow run metadata available)
2025-12-05 15:26:20,766 - INFO - Total experiments: 70
2025-12-05 15:26:20,769 - INFO - Results saved to: C:\Users\Lenovo\Documents\projects\research\Major-Project\Forest-Audio\data\processed_audio\mode